In [ ]:
import pandas as pd
import math
import matplotlib.pyplot as plt

df = pd.read_csv("../data/ab_test_data.csv")

print(df.head())
print()
print(df.shape)

In [ ]:
combined_analysis = (
    df.groupby(["device", "traffic_source", "group"])
    .agg(
        users=("user_id", "count"),
        conversions=("converted", "sum"),
        conversion_rate=("converted", "mean")
    )
    .reset_index()
)

combined_analysis["conversion_rate_pct"] = (
    combined_analysis["conversion_rate"] * 100
)

print(combined_analysis.round(4))

In [ ]:
combined_pivot = combined_analysis.pivot_table(
    index=["device", "traffic_source"],
    columns="group",
    values="conversion_rate"
)

combined_pivot["absolute_lift"] = (
    combined_pivot["treatment"] - combined_pivot["control"]
)

combined_pivot["relative_lift"] = (
    combined_pivot["absolute_lift"] / combined_pivot["control"]
)

combined_pivot["control_pct"] = combined_pivot["control"] * 100
combined_pivot["treatment_pct"] = combined_pivot["treatment"] * 100
combined_pivot["lift_pp"] = combined_pivot["absolute_lift"] * 100
combined_pivot["relative_lift_pct"] = combined_pivot["relative_lift"] * 100

segment_lift = combined_pivot[
    [
        "control_pct",
        "treatment_pct",
        "lift_pp",
        "relative_lift_pct"
    ]
].sort_values("lift_pp", ascending=False)

print(segment_lift.round(2))

In [ ]:
segment_sizes = (
    combined_analysis
    .pivot_table(
        index=["device", "traffic_source"],
        columns="group",
        values="users"
    )
)

segment_summary = segment_lift.copy()

segment_summary["control_users"] = segment_sizes["control"]
segment_summary["treatment_users"] = segment_sizes["treatment"]

segment_summary = segment_summary[
    [
        "control_users",
        "treatment_users",
        "control_pct",
        "treatment_pct",
        "lift_pp",
        "relative_lift_pct"
    ]
]

print(segment_summary.round(2))

In [ ]:
combined_results = []

for (device, source), segment in df.groupby(["device", "traffic_source"]):

    control = segment[segment["group"] == "control"]["converted"]
    treatment = segment[segment["group"] == "treatment"]["converted"]

    control_users = len(control)
    treatment_users = len(treatment)

    control_conversions = control.sum()
    treatment_conversions = treatment.sum()

    control_rate = control.mean()
    treatment_rate = treatment.mean()

    pooled_rate = (
        control_conversions + treatment_conversions
    ) / (
        control_users + treatment_users
    )

    standard_error = math.sqrt(
        pooled_rate * (1 - pooled_rate) *
        ((1 / control_users) + (1 / treatment_users))
    )

    z_stat = (control_rate - treatment_rate) / standard_error

    p_value = 0.5 * (
        1 + math.erf(z_stat / math.sqrt(2))
    )

    combined_results.append({
        "device": device,
        "traffic_source": source,
        "control_users": control_users,
        "treatment_users": treatment_users,
        "lift_pp": (treatment_rate - control_rate) * 100,
        "p_value": p_value,
        "significant": p_value < 0.05
    })

combined_test_results = pd.DataFrame(combined_results)

combined_test_results = combined_test_results.sort_values(
    "p_value"
)

print(combined_test_results.round(4))

In [ ]:
number_of_tests = len(combined_test_results)

bonferroni_alpha = 0.05 / number_of_tests

combined_test_results["bonferroni_significant"] = (
    combined_test_results["p_value"] < bonferroni_alpha
)

print(f"Number of Tests: {number_of_tests}")
print(f"Original Alpha: 0.0500")
print(f"Bonferroni Adjusted Alpha: {bonferroni_alpha:.4f}")
print()

print(
    combined_test_results[
        [
            "device",
            "traffic_source",
            "lift_pp",
            "p_value",
            "bonferroni_significant"
        ]
    ].round(4)
)

## Multiple Testing Correction

The combined device and traffic-source analysis tested 12 different customer segments.

Before adjustment, the Tablet + Social segment appeared statistically significant with a p-value of 0.0285. However, testing many segments increases the probability of finding a significant result by chance.

Using the Bonferroni correction, the significance threshold was adjusted from 0.05 to approximately 0.0042.

After applying this correction, none of the combined segments remained statistically significant.

Therefore, the apparent Tablet + Social effect should be treated as exploratory rather than strong evidence of a true treatment effect.

In [ ]:
overall_summary = (
    df.groupby("group")
    .agg(
        users=("user_id", "count"),
        conversions=("converted", "sum"),
        conversion_rate=("converted", "mean")
    )
)

control_rate = overall_summary.loc["control", "conversion_rate"]
treatment_rate = overall_summary.loc["treatment", "conversion_rate"]

absolute_lift = treatment_rate - control_rate
relative_lift = absolute_lift / control_rate

summary = pd.DataFrame({
    "Metric": [
        "Control Users",
        "Treatment Users",
        "Control Conversion Rate",
        "Treatment Conversion Rate",
        "Absolute Lift",
        "Relative Lift"
    ],
    "Value": [
        f"{overall_summary.loc['control', 'users']:,}",
        f"{overall_summary.loc['treatment', 'users']:,}",
        f"{control_rate:.2%}",
        f"{treatment_rate:.2%}",
        f"{absolute_lift:.2%}",
        f"{relative_lift:.2%}"
    ]
})

print(summary.to_string(index=False))